# RetainIQ — Phase 1.3: Business Field Audit

## Objective

Translate technical observations into business rules for the RetainIQ analytical pipeline.

This notebook examines:

- Customer outcomes
- Churn-detail applicability
- Offer and internet-service semantics
- Core service and account categories
- Customer value and satisfaction measures
- New-customer identification
- Downstream modeling considerations

### Audit principle

**Business meaning determines transformation rules.**

## 1. Environment Setup and Data Ingestion

The raw dataset is reloaded independently so this notebook can be audited or executed on its own.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Project convention:
# RetainIQ/
# ├── data/
# │   └── raw/
# │       └── telco.csv
# └── phase_01_data_audit/
#     └── notebooks/
#
# From this notebook, the raw dataset is two levels up from the phase folder.
DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\telco.csv")

# Fallback for running the notebook alongside the uploaded file during development.
if not DATA_PATH.exists():
    DATA_PATH = Path("telco.csv")

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

Loaded: 7,043 rows × 50 columns


## 2. Customer Outcome Structure

`Customer Status` and `Churn Label` are both outcome-related fields. Their individual category
distributions are inspected before checking whether the two fields agree.

In [2]:
for col in ["Customer Status", "Churn Label"]:
    print(f"\n{'=' * 70}")
    print(col)
    print(f"{'=' * 70}")
    print(df[col].value_counts(dropna=False))


Customer Status
Customer Status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64

Churn Label
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64


## 3. Customer Status × Churn Label Validation

A cross-tab makes the relationship between the two outcome fields explicit.

In [3]:
status_churn = pd.crosstab(
    df["Customer Status"],
    df["Churn Label"],
    margins=True
)

status_churn

Churn Label,No,Yes,All
Customer Status,,,
Churned,0,1869,1869
Joined,454,0,454
Stayed,4720,0,4720
All,5174,1869,7043


In [4]:
status_churn_detail = pd.crosstab(
    df["Customer Status"],
    df["Churn Label"]
)

assert status_churn_detail.loc["Churned", "Yes"] == 1869
assert status_churn_detail.loc["Stayed", "No"] == 4720
assert status_churn_detail.loc["Joined", "No"] == 454

print("PASS — Customer Status and Churn Label are internally consistent.")

PASS — Customer Status and Churn Label are internally consistent.


### Interpretation

The source contains:

- **1,869 Churned** customers with `Churn Label = Yes`
- **4,720 Stayed** customers with `Churn Label = No`
- **454 Joined** customers with `Churn Label = No`

The `Joined` population is analytically important because these customers are new and should
not be treated as equivalent to customers with a completed historical stay/churn outcome.

## 4. Churn Category and Churn Reason Applicability

Churn category and reason are conditional business fields. We therefore test their missingness
against the churn label rather than treating nulls as generic data-quality failures.

In [5]:
churn_detail_audit = (
    df.groupby("Churn Label")
      .agg(
          customers=("Customer ID", "count"),
          churn_category_nulls=("Churn Category", lambda s: s.isna().sum()),
          churn_reason_nulls=("Churn Reason", lambda s: s.isna().sum())
      )
)

churn_detail_audit

,customers,churn_category_nulls,churn_reason_nulls
Churn Label,,,
No,5174,5174,5174
Yes,1869,0,0


In [6]:
pd.crosstab(
    df["Churn Label"],
    df["Churn Category"].isna(),
    margins=True
)

Churn Category,False,True,All
Churn Label,,,
No,0,5174,5174
Yes,1869,0,1869
All,1869,5174,7043


### Interpretation

The missing churn-detail values are structurally associated with customers who did not churn.
Therefore:

> `Churn Category` and `Churn Reason` should **remain null** for non-churned customers.

Filling these fields with arbitrary labels would erase the distinction between "not applicable"
and "unknown".

## 5. Offer and Internet-Type Semantics

The remaining major nulls are inspected alongside their related service fields.
The objective is to determine whether a null can reasonably be represented as an explicit
business state.

In [7]:
for col in ["Offer", "Internet Type", "Internet Service"]:
    print(f"\n{'=' * 70}")
    print(col)
    print(f"{'=' * 70}")
    print(df[col].value_counts(dropna=False))


Offer
Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64

Internet Type
Internet Type
Fiber Optic    3035
DSL            1652
NaN            1526
Cable           830
Name: count, dtype: int64

Internet Service
Internet Service
Yes    5517
No     1526
Name: count, dtype: int64


### Interpretation

`Offer` nulls can be standardized to an explicit **`No Offer`** category.

`Internet Type` nulls can be standardized to **`No Internet Service`**, making the non-applicable
state explicit and easier to group and filter.

These are semantic normalizations rather than statistical imputations.

## 6. Core Customer-Service and Account Fields

Key categorical dimensions are profiled because they will later support churn-driver analysis,
segmentation, SQL dimensions, and dashboard filters.

In [8]:
business_fields = [
    "Contract",
    "Payment Method",
    "Internet Service",
    "Online Security",
    "Premium Tech Support",
    "Paperless Billing",
    "Senior Citizen",
    "Dependents"
]

for col in business_fields:
    print(f"\n{'=' * 70}\n{col}\n{'=' * 70}")
    print(df[col].value_counts(dropna=False))


Contract
Contract
Month-to-Month    3610
Two Year          1883
One Year          1550
Name: count, dtype: int64

Payment Method
Payment Method
Bank Withdrawal    3909
Credit Card        2749
Mailed Check        385
Name: count, dtype: int64

Internet Service
Internet Service
Yes    5517
No     1526
Name: count, dtype: int64

Online Security
Online Security
No     5024
Yes    2019
Name: count, dtype: int64

Premium Tech Support
Premium Tech Support
No     4999
Yes    2044
Name: count, dtype: int64

Paperless Billing
Paperless Billing
Yes    4171
No     2872
Name: count, dtype: int64

Senior Citizen
Senior Citizen
No     5901
Yes    1142
Name: count, dtype: int64

Dependents
Dependents
No     5416
Yes    1627
Name: count, dtype: int64


The important point is not merely the number of categories. These fields represent potential
customer-risk dimensions and will later be compared against churn and customer value.

## 7. Customer Value, Satisfaction, and Risk Fields

The project contains several measures that look related but represent different concepts.
They should be kept distinct in downstream analysis.

In [9]:
value_fields = [
    "Monthly Charge",
    "Total Charges",
    "Total Revenue",
    "CLTV",
    "Churn Score",
    "Satisfaction Score"
]

df[value_fields].describe().T

,count,mean,std,min,25%,50%,75%,max
Monthly Charge,"7,043.00",64.76,30.09,18.25,35.50,70.35,89.85,118.75
Total Charges,"7,043.00","2,280.38","2,266.22",18.80,400.15,"1,394.55","3,786.60","8,684.80"
Total Revenue,"7,043.00","3,034.38","2,865.20",21.36,605.61,"2,108.64","4,801.15","11,979.34"
CLTV,"7,043.00","4,400.30","1,183.06","2,003.00","3,469.00","4,527.00","5,380.50","6,500.00"
Churn Score,"7,043.00",58.51,21.17,5.00,40.00,61.00,75.50,96.00
Satisfaction Score,"7,043.00",3.24,1.20,1.00,3.00,3.00,4.00,5.00


### Interpretation

The fields have different analytical meanings:

- `Monthly Charge`: recurring customer charge
- `Total Charges`: accumulated charges
- `Total Revenue`: supplied revenue measure
- `CLTV`: supplied customer-value metric
- `Churn Score`: supplied risk score
- `Satisfaction Score`: customer-experience metric

Later analysis should avoid treating these fields as interchangeable proxies.

## 8. New-Customer Population and Modeling Eligibility

New customers are explicitly measured because they require different treatment from customers
with an observed churn/stay history.

In [10]:
new_customer_audit = pd.Series({
    "Joined Customers": int(df["Customer Status"].eq("Joined").sum()),
    "Share of Dataset (%)": round(df["Customer Status"].eq("Joined").mean() * 100, 2),
    "Joined With Churn Label = Yes": int(
        ((df["Customer Status"] == "Joined") & (df["Churn Label"] == "Yes")).sum()
    ),
})

new_customer_audit

Joined Customers                454.00
Share of Dataset (%)              6.45
Joined With Churn Label = Yes     0.00
dtype: float64

In [13]:
joined_sample = df.loc[
    df["Customer Status"] == "Joined",
    ["Customer ID", "Customer Status", "Churn Label"]
].head(10)

joined_sample

,Customer ID,Customer Status,Churn Label
477,4929-XIHVW,Joined,No
478,3413-BMNZE,Joined,No
486,2424-WVHPL,Joined,No
508,0021-IKXGC,Joined,No
514,0224-RLWWD,Joined,No
559,3966-HRMZA,Joined,No
642,4854-CIDCF,Joined,No
644,6407-UTSLV,Joined,No
670,8775-ERLNB,Joined,No
671,8309-IEYJD,Joined,No


### Modeling decision

Create `is_new_customer = (Customer Status == "Joined")` during Phase 2.

The churn-modeling population should exclude these new customers from the historical churn/stay
training population, following the project roadmap.

## 9. Business Decision Log

The audit findings are now converted into explicit transformation decisions. This table is the
handoff between discovery and implementation.

In [14]:
decision_log = pd.DataFrame([
    {
        "Finding": "Missing Offer",
        "Evidence": f"{df['Offer'].isna().sum():,} nulls",
        "Decision": 'Replace with "No Offer"',
        "Rationale": "Represents an explicit no-offer state."
    },
    {
        "Finding": "Missing Internet Type",
        "Evidence": f"{df['Internet Type'].isna().sum():,} nulls",
        "Decision": 'Replace with "No Internet Service"',
        "Rationale": "Makes the non-applicable service state explicit."
    },
    {
        "Finding": "Churn Category / Reason nulls",
        "Evidence": f"{df['Churn Category'].isna().sum():,} nulls each",
        "Decision": "Preserve nulls",
        "Rationale": "Structurally not applicable to non-churned customers."
    },
    {
        "Finding": "Joined customers",
        "Evidence": f"{df['Customer Status'].eq('Joined').sum():,} customers",
        "Decision": "Create is_new_customer",
        "Rationale": "Separates new customers from historical churn/stay outcomes."
    },
    {
        "Finding": "String whitespace",
        "Evidence": "Observed leading/trailing whitespace in categorical fields",
        "Decision": "Strip string whitespace defensively",
        "Rationale": "Protects categorical consistency in downstream processing."
    }
])

decision_log

,Finding,Evidence,Decision,Rationale
0,Missing Offer,"3,877 nulls","Replace with ""No Offer""",Represents an explicit no-offer state.
1,Missing Internet Type,"1,526 nulls","Replace with ""No Internet Service""",Makes the non-applicable service state explicit.
2,Churn Category / Reason nulls,"5,174 nulls each",Preserve nulls,Structurally not applicable to non-churned cus...
3,Joined customers,454 customers,Create is_new_customer,Separates new customers from historical churn/...
4,String whitespace,Observed leading/trailing whitespace in catego...,Strip string whitespace defensively,Protects categorical consistency in downstream...


## 10. Notebook 3 Conclusion

The business-field audit establishes the rules required for safe cleaning:

- Some nulls represent explicit business states.
- Some nulls are structurally conditional and must remain null.
- `Customer Status` defines a distinct `Joined` population that needs a downstream flag.
- Customer value, satisfaction, and churn-risk measures must retain their separate meanings.

**Next notebook:** `04_audit_summary.ipynb`